### Dataset: Energy Efficiency with CL (cooling load) as target

In [1]:
from ucimlrepo import fetch_ucirepo

energy_efficiency = fetch_ucirepo(id=242) 
  
X = energy_efficiency.data.features 
y = energy_efficiency.data.targets

var_df = energy_efficiency.variables
col_map = dict(zip(var_df["name"], var_df["description"]))
X = X.rename(columns=col_map)
y = y.rename(columns=col_map)

y = y[["Cooling Load"]]
df = X.join(y)
df.head()

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Orientation,Glazing Area,Glazing Area Distribution,Cooling Load
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,28.28


### Train test split

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,     
    random_state=42,   
    shuffle=True
)

print(f"#X_train = {len(X_train)}   #y_train = {len(y_train)}")
print(f"#X_test = {len(X_test)}     #y_test = {len(y_test)}")

#X_train = 691   #y_train = 691
#X_test = 77     #y_test = 77


### EDF normalization

In [3]:
from src.edf import edf_normalize, col_denorm

X_train_norm, X_test_norm, _ = edf_normalize(X_train, X_test)
y_train_norm, y_test_norm, edf_models = edf_normalize(y_train, y_test)

y_denorm = col_denorm("Cooling Load", edf_models)

### Basic evaluation for different features and target degrees

In [ ]:
from src.features import moment_like_features, prepare_targets
from src.hcr import fit_lasso, plot_example_densities
from src.evaluation import mean_log_likelihood, mse_evaluation
from collections import defaultdict

N_vals = [n for n in range(2, 9)]
A = 3.
B=100.

ll_softplus_vals = defaultdict(dict)
ll_param_softplus_vals = defaultdict(dict)
ll_clip_vals = defaultdict(dict)

MSE_softplus_vals = defaultdict(dict)
MSE_param_softplus_vals = defaultdict(dict)
MSE_clip_vals = defaultdict(dict)

for n_feature in N_vals:
    for n_target in N_vals:
        print(f"----- Target deg. {n_target}, feature deg. {n_feature} -----")

        V_train = moment_like_features(X_train_norm, n_feature)
        V_test  = moment_like_features(X_test_norm, n_feature)

        targets_train = prepare_targets(y_train_norm, n_target)
        targets_test  = prepare_targets(y_test_norm, n_target)

        models = []
        for n in range(n_target):
            models.append(fit_lasso(V_train, targets_train[n]))

        plot_example_densities(V_test, y_test_norm, models,
                               name=f"Example densities (\"softplus\", features deg. {n_feature}, target deg. {n_target})",
                               method="softplus", seed=42, save=True)
        plot_example_densities(V_test, y_test_norm, models,
                               name=f"Example densities (\"param-softplus\", features deg. {n_feature}, target deg. {n_target})",
                               method="softplus", 
                               a=A, b=B, seed=42, save=True)
        plot_example_densities(V_test, y_test_norm, models,
                               name=f"Example densities (\"clip\", features deg. {n_feature}, target deg. {n_target})",
                               method="clip", seed=42, save=True)
        
        ll_softplus = mean_log_likelihood(V_test, y_test_norm, models)
        ll_param_softplus = mean_log_likelihood(V_test, y_test_norm, models,
                                                method="softplus", a=A, b=B)
        ll_clip = mean_log_likelihood(V_test, y_test_norm, models, method="clip")

        mse_softplus = mse_evaluation(V_test, y_test, models, y_denorm)
        mse_param_softplus = mse_evaluation(V_test, y_test, models, y_denorm,
                                            method="softplus", a=A, b=B)
        mse_clip = mse_evaluation(V_test, y_test, models, y_denorm, method="clip")
        
        ll_softplus_vals[n_target][n_feature] = ll_softplus
        ll_param_softplus_vals[n_target][n_feature] = ll_param_softplus
        ll_clip_vals[n_target][n_feature] = ll_clip

        MSE_softplus_vals[n_target][n_feature] = mse_softplus
        MSE_param_softplus_vals[n_target][n_feature] = mse_param_softplus
        MSE_clip_vals[n_target][n_feature] = mse_clip

In [5]:
for n_feature in N_vals:
    for n_target in N_vals:
        print(f"Features deg. {n_feature}, Target deg. {n_target}:")

        print(f"  Mean log-likelihood (softplus): {ll_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  Mean log-likelihood (param-softplus): {ll_param_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  Mean log-likelihood (clip):     {ll_clip_vals[n_target][n_feature]:.4f}")

        print(f"  MSE (softplus):                 {MSE_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  MSE (param-softplus):                 {MSE_param_softplus_vals[n_target][n_feature]:.4f}")
        print(f"  MSE (clip):                     {MSE_clip_vals[n_target][n_feature]:.4f}")

Features deg. 2, Target deg. 2:
  Mean log-likelihood (softplus): 0.4665
  Mean log-likelihood (param-softplus): 0.7541
  Mean log-likelihood (clip):     0.6574
  MSE (softplus):                 26.5800
  MSE (param-softplus):                 13.4264
  MSE (clip):                     14.6552
Features deg. 2, Target deg. 3:
  Mean log-likelihood (softplus): 0.5717
  Mean log-likelihood (param-softplus): 0.8943
  Mean log-likelihood (clip):     0.7890
  MSE (softplus):                 26.3923
  MSE (param-softplus):                 11.2125
  MSE (clip):                     14.6265
Features deg. 2, Target deg. 4:
  Mean log-likelihood (softplus): 0.6359
  Mean log-likelihood (param-softplus): 0.9816
  Mean log-likelihood (clip):     0.8445
  MSE (softplus):                 27.4076
  MSE (param-softplus):                 10.9922
  MSE (clip):                     17.5425
Features deg. 2, Target deg. 5:
  Mean log-likelihood (softplus): 0.6713
  Mean log-likelihood (param-softplus): 0.9941
 

In [6]:
def process_results(results_dict, reverse=True):
    sorted_items = sorted(
        ((n_target, n_features, val) for n_target, inner in results_dict.items() for n_features, val in inner.items()),
        key=lambda x: x[2],
        reverse=reverse
    )
    return sorted_items

In [7]:
def print_best_results(results_dict, results_type="", cal_method="", N_vals=10, reverse=True):
    sorted = process_results(results_dict, reverse=reverse)
    print(f"Best {results_type} values for \"{cal_method}\" calibration:")
    for id in range(N_vals):
        target_deg, feature_deg, val = sorted[id]
        print(f"{id+1}. Target deg: {target_deg}, feature deg: {feature_deg}")
        print(f"    {val:.4f}")

In [8]:
print_best_results(ll_softplus_vals, 
                   results_type="mean log-likelihood",
                   cal_method="softplus")

Best mean log-likelihood values for "softplus" calibration:
1. Target deg: 8, feature deg: 8
    0.9256
2. Target deg: 8, feature deg: 7
    0.9240
3. Target deg: 8, feature deg: 6
    0.9187
4. Target deg: 8, feature deg: 5
    0.9167
5. Target deg: 7, feature deg: 8
    0.9096
6. Target deg: 7, feature deg: 7
    0.9081
7. Target deg: 7, feature deg: 6
    0.9031
8. Target deg: 7, feature deg: 5
    0.9015
9. Target deg: 8, feature deg: 4
    0.8978
10. Target deg: 7, feature deg: 4
    0.8894


In [9]:
print_best_results(MSE_softplus_vals, 
                   results_type="mean squared error",
                   cal_method="softplus",
                   reverse=False)

Best mean squared error values for "softplus" calibration:
1. Target deg: 3, feature deg: 8
    21.8579
2. Target deg: 3, feature deg: 7
    21.9105
3. Target deg: 3, feature deg: 6
    22.0397
4. Target deg: 3, feature deg: 5
    22.0909
5. Target deg: 4, feature deg: 8
    22.5312
6. Target deg: 4, feature deg: 7
    22.5803
7. Target deg: 5, feature deg: 8
    22.6683
8. Target deg: 5, feature deg: 7
    22.7293
9. Target deg: 3, feature deg: 4
    22.8042
10. Target deg: 4, feature deg: 6
    22.8052


In [10]:
print_best_results(ll_param_softplus_vals, 
                   results_type="mean log-likelihood",
                   cal_method="param-softplus")

Best mean log-likelihood values for "param-softplus" calibration:
1. Target deg: 8, feature deg: 8
    1.3624
2. Target deg: 8, feature deg: 7
    1.3617
3. Target deg: 8, feature deg: 6
    1.3583
4. Target deg: 8, feature deg: 5
    1.3568
5. Target deg: 8, feature deg: 4
    1.3408
6. Target deg: 7, feature deg: 8
    1.3276
7. Target deg: 7, feature deg: 7
    1.3264
8. Target deg: 7, feature deg: 6
    1.3205
9. Target deg: 7, feature deg: 5
    1.3189
10. Target deg: 7, feature deg: 4
    1.3136


In [13]:
print_best_results(MSE_param_softplus_vals, 
                   results_type="mean squared error",
                   cal_method="param-softplus",
                   reverse=False)

Best mean squared error values for "param-softplus" calibration:
1. Target deg: 6, feature deg: 7
    5.8467
2. Target deg: 6, feature deg: 5
    5.8481
3. Target deg: 6, feature deg: 8
    5.8545
4. Target deg: 6, feature deg: 6
    5.8580
5. Target deg: 6, feature deg: 4
    5.9125
6. Target deg: 8, feature deg: 6
    6.1730
7. Target deg: 8, feature deg: 5
    6.1971
8. Target deg: 8, feature deg: 7
    6.2130
9. Target deg: 8, feature deg: 8
    6.2222
10. Target deg: 4, feature deg: 7
    6.3175


In [11]:
print_best_results(ll_clip_vals, 
                   results_type="mean log-likelihood",
                   cal_method="clip")

Best mean log-likelihood values for "clip" calibration:
1. Target deg: 8, feature deg: 8
    1.1452
2. Target deg: 8, feature deg: 7
    1.1437
3. Target deg: 8, feature deg: 6
    1.1374
4. Target deg: 7, feature deg: 8
    1.1366
5. Target deg: 7, feature deg: 7
    1.1354
6. Target deg: 8, feature deg: 5
    1.1352
7. Target deg: 7, feature deg: 6
    1.1287
8. Target deg: 7, feature deg: 5
    1.1268
9. Target deg: 7, feature deg: 4
    1.1143
10. Target deg: 8, feature deg: 4
    1.1136


In [12]:
print_best_results(MSE_clip_vals, 
                   results_type="mean squared error",
                   cal_method="clip",
                   reverse=False)

Best mean squared error values for "clip" calibration:
1. Target deg: 3, feature deg: 8
    10.1591
2. Target deg: 3, feature deg: 7
    10.1718
3. Target deg: 3, feature deg: 6
    10.1891
4. Target deg: 3, feature deg: 5
    10.1925
5. Target deg: 4, feature deg: 8
    10.4960
6. Target deg: 4, feature deg: 7
    10.5109
7. Target deg: 4, feature deg: 6
    10.5404
8. Target deg: 4, feature deg: 5
    10.5599
9. Target deg: 4, feature deg: 4
    10.6529
10. Target deg: 3, feature deg: 4
    10.8765


### CV for top basic results

In [14]:
from src.cv import cross_validate, print_cv_results

def cv_top(X, y, results_dict, 
           results_type, method,
           a=1, b=1, eps=1e-6,
           N_vals=5, reverse=True):
    sorted = process_results(results_dict, reverse=reverse)
    results = []
    print(f"CV for top {results_type} values with \"{method}\" calibration:")
    if method=="param-softplus": method="softplus"

    for id in range(N_vals):
        target_deg, feature_deg, _ = sorted[id]
        print(f"{id+1}. Target deg: {target_deg}, feature deg: {feature_deg}")
        ll_results, mse_results = cross_validate(X, y,
                                                 feature_deg, target_deg,
                                                 method=method,
                                                 a=a, b=b, eps=eps)
        results.append({"target_deg": target_deg,
                        "feature_deg": feature_deg,
                        "ll_results": ll_results,
                        "mse_results": mse_results})
        print_cv_results(ll_results, mse_results)
    return results

In [15]:
cv_ll_softplus_results = cv_top(X, y, ll_softplus_vals,
                                results_type="mean log-likelihood",
                                method="softplus")

CV for top mean log-likelihood values with "softplus" calibration:
1. Target deg: 8, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.87 0.96 0.84 0.87 0.91 0.9  0.83 0.83 0.81 0.84]
  mean LL : 0.8657
  std LL  : 0.0426

Mean square error:
  per fold: [14.3  23.32 12.34 28.33 26.53 11.2  15.31 26.72 16.42 12.75]
  mean MSE: 18.7240
  std MSE : 6.3848
2. Target deg: 8, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.87 0.96 0.84 0.87 0.91 0.9  0.83 0.83 0.81 0.84]
  mean LL : 0.8651
  std LL  : 0.0422

Mean square error:
  per fold: [14.28 23.25 12.38 28.33 26.54 11.2  15.29 26.74 16.4  12.78]
  mean MSE: 18.7197
  std MSE : 6.3804
3. Target deg: 8, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.87 0.95 0.83 0.87 0.9  0.89 0.83 0.83 0.81 0.83]
  mean LL : 0.8616
  std LL  : 0.041

In [16]:
cv_mse_softplus_results = cv_top(X, y, MSE_softplus_vals,
                                 results_type="mean squared error",
                                 method="softplus",
                                 reverse=False)

CV for top mean squared error values with "softplus" calibration:
1. Target deg: 3, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.64 0.68 0.59 0.69 0.7  0.63 0.57 0.62 0.63 0.57]
  mean LL : 0.6323
  std LL  : 0.0453

Mean square error:
  per fold: [13.26 21.26 11.74 25.89 23.68 10.39 12.79 24.54 15.06 11.39]
  mean MSE: 17.0002
  std MSE : 5.8047
2. Target deg: 3, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.64 0.68 0.59 0.69 0.7  0.63 0.57 0.62 0.63 0.57]
  mean LL : 0.6320
  std LL  : 0.0453

Mean square error:
  per fold: [13.28 21.3  11.75 25.95 23.7  10.41 12.81 24.52 15.1  11.42]
  mean MSE: 17.0247
  std MSE : 5.8048
3. Target deg: 3, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.64 0.68 0.59 0.69 0.7  0.62 0.56 0.62 0.63 0.57]
  mean LL : 0.6300
  std LL  : 0.0447

In [17]:
cv_ll_param_softplus_results = cv_top(X, y, ll_param_softplus_vals,
                                      results_type="mean log-likelihood",
                                      method="param-softplus",
                                      a=3., b=100.)

CV for top mean log-likelihood values with "param-softplus" calibration:
1. Target deg: 8, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.23 1.47 1.24 1.25 1.31 1.31 1.14 1.25 1.04 1.26]
  mean LL : 1.2501
  std LL  : 0.1048

Mean square error:
  per fold: [4.68 8.08 3.29 8.18 8.7  4.65 7.21 8.99 6.94 3.12]
  mean MSE: 6.3837
  std MSE : 2.1304
2. Target deg: 8, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.23 1.47 1.24 1.25 1.31 1.32 1.14 1.25 1.05 1.26]
  mean LL : 1.2518
  std LL  : 0.1042

Mean square error:
  per fold: [4.67 8.19 3.31 8.13 8.66 4.62 7.21 9.   6.95 3.17]
  mean MSE: 6.3896
  std MSE : 2.1252
3. Target deg: 8, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.23 1.47 1.24 1.26 1.31 1.32 1.15 1.25 1.05 1.26]
  mean LL : 1.2514
  std LL  : 0.1028

Mean square e

In [18]:
cv_mse_param_softplus_results = cv_top(X, y, MSE_param_softplus_vals,
                                       results_type="mean squared error",
                                       method="param-softplus",
                                       a=3., b=100.,
                                       reverse=False)

CV for top mean squared error values with "param-softplus" calibration:
1. Target deg: 6, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.2  1.38 1.17 1.18 1.28 1.21 0.97 1.21 1.   1.19]
  mean LL : 1.1801
  std LL  : 0.1127

Mean square error:
  per fold: [4.76 7.65 3.73 7.35 8.27 5.05 7.29 8.47 7.57 3.38]
  mean MSE: 6.3507
  std MSE : 1.8229
2. Target deg: 6, feature deg: 5
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.19 1.38 1.17 1.18 1.27 1.21 0.97 1.21 1.   1.19]
  mean LL : 1.1769
  std LL  : 0.1116

Mean square error:
  per fold: [4.76 7.79 3.62 7.42 8.21 4.99 7.24 8.57 7.52 3.49]
  mean MSE: 6.3636
  std MSE : 1.8395
3. Target deg: 6, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.2  1.38 1.17 1.18 1.28 1.21 0.97 1.21 1.   1.19]
  mean LL : 1.1800
  std LL  : 0.1137

Mean square er

In [19]:
cv_ll_clip_results = cv_top(X, y, ll_clip_vals,
                            results_type="mean log-likelihood",
                            method="clip")

CV for top mean log-likelihood values with "clip" calibration:
1. Target deg: 8, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.1  1.19 1.07 1.09 0.97 1.13 1.04 1.05 1.   0.9 ]
  mean LL : 1.0516
  std LL  : 0.0790

Mean square error:
  per fold: [ 8.15 16.37  6.35 17.6  15.98  6.35  9.16 16.85  9.38  6.29]
  mean MSE: 11.2476
  std MSE : 4.5881
2. Target deg: 8, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.09 1.19 1.06 1.09 0.97 1.13 1.04 1.05 1.   0.9 ]
  mean LL : 1.0511
  std LL  : 0.0790

Mean square error:
  per fold: [ 8.16 16.32  6.36 17.64 15.94  6.34  9.15 16.87  9.37  6.31]
  mean MSE: 11.2475
  std MSE : 4.5843
3. Target deg: 8, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [1.09 1.19 1.06 1.08 0.96 1.12 1.04 1.04 1.   0.89]
  mean LL : 1.0476
  std LL  : 0.0787

M

In [20]:
cv_mse_clip_results = cv_top(X, y, MSE_clip_vals,
                             results_type="mean squared error",
                             method="clip",
                             reverse=False)

CV for top mean squared error values with "clip" calibration:
1. Target deg: 3, feature deg: 8
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.86 0.9  0.81 0.91 0.92 0.84 0.76 0.83 0.84 0.78]
  mean LL : 0.8453
  std LL  : 0.0504

Mean square error:
  per fold: [ 5.96 12.14  4.91 13.37 11.07  5.23  5.44 12.47  6.79  5.26]
  mean MSE: 8.2640
  std MSE : 3.3401
2. Target deg: 3, feature deg: 7
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.86 0.9  0.81 0.91 0.92 0.84 0.76 0.83 0.83 0.78]
  mean LL : 0.8450
  std LL  : 0.0504

Mean square error:
  per fold: [ 5.97 12.23  4.94 13.33 11.06  5.24  5.48 12.47  6.79  5.29]
  mean MSE: 8.2810
  std MSE : 3.3334
3. Target deg: 3, feature deg: 6
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Fold 6
Fold 7
Fold 8
Fold 9
Fold 10

Log-likelihood:
  per fold: [0.85 0.9  0.81 0.91 0.92 0.84 0.76 0.83 0.83 0.78]
  mean LL : 0.8433
  std LL  : 0.0499

Mean

### Results summary

In [21]:
import pandas as pd
import numpy as np

def results_to_df(results):
    rows = []
    for r in results:
        rows.append({
            "target_deg": r["target_deg"],
            "feature_deg": r["feature_deg"],
            "ll_mean": np.mean(r["ll_results"]),
            "ll_std": np.std(r["ll_results"]),
            "mse_mean": np.mean(r["mse_results"]),
            "mse_std": np.std(r["mse_results"]),
        })
    return pd.DataFrame(rows)

In [22]:
results = {
    "ll_softplus": cv_ll_softplus_results,
    "mse_softplus": cv_mse_softplus_results,

    "ll_param_softplus": cv_ll_param_softplus_results,
    "mse_param_softplus": cv_mse_param_softplus_results,

    "ll_clip": cv_ll_clip_results,
    "mse_clip": cv_mse_clip_results,
}

In [23]:
results_dfs = {key : results_to_df(val) for key, val in results.items()}

In [24]:
df = results_dfs["ll_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,8,8,0.865711,0.042612,18.723983,6.384758
1,8,7,0.865104,0.042201,18.719739,6.380386
2,8,6,0.861592,0.041805,18.794046,6.383271
3,8,5,0.860765,0.041353,18.807253,6.386179
4,7,8,0.849216,0.037479,18.411500,6.343164


In [25]:
df = results_dfs["mse_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,3,8,0.632276,0.045334,17.000220,5.804687
1,3,7,0.631966,0.045271,17.024711,5.804751
2,3,6,0.630044,0.044658,17.146127,5.845738
3,3,5,0.629533,0.044387,17.138796,5.832739
4,4,8,0.711605,0.040296,17.664294,5.890629


In [26]:
df = results_dfs["ll_param_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,8,8,1.250115,0.104765,6.383664,2.130431
1,8,7,1.251786,0.104190,6.389639,2.125194
2,8,6,1.251416,0.102838,6.390376,2.131331
3,8,5,1.251684,0.102201,6.393445,2.123465
4,8,4,1.245309,0.097732,6.596561,2.189257


In [27]:
df = results_dfs["mse_param_softplus"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,6,7,1.180144,0.112731,6.350692,1.822882
1,6,5,1.176853,0.111598,6.363592,1.839451
2,6,8,1.179964,0.113711,6.349332,1.816907
3,6,6,1.176922,0.112004,6.361089,1.839448
4,6,4,1.171349,0.110882,6.399213,1.877627


In [28]:
df = results_dfs["ll_clip"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,8,8,1.051642,0.079037,11.247649,4.588138
1,8,7,1.051098,0.079031,11.247478,4.584273
2,8,6,1.047565,0.078733,11.250500,4.586655
3,7,8,1.070490,0.040446,10.941380,4.536264
4,7,7,1.069995,0.040094,10.937768,4.522419


In [29]:
df = results_dfs["mse_clip"]
df

,target_deg,feature_deg,ll_mean,ll_std,mse_mean,mse_std
0,3,8,0.845258,0.050431,8.264006,3.340057
1,3,7,0.844957,0.050396,8.280990,3.333412
2,3,6,0.843310,0.049867,8.356396,3.363882
3,3,5,0.842966,0.049643,8.340410,3.338796
4,4,8,0.919799,0.055547,8.365122,3.118083
